# Bulk 2D image generator — Google Colab

Generate one PNG for each numbered prompt in `image_prompts.txt` using an open-source Diffusers model. **Runtime → Change runtime type → T4 GPU** first. Run cells top to bottom. Output goes to your Google Drive, so interrupted runs resume without redoing completed images. No API key is needed for the default public model. Generation is not unlimited: Colab GPU availability, runtime duration and Drive space apply.

**Note:** AI text on signs/tags may be misspelled; review outputs. The default model does not directly reproduce uploaded reference pictures. Use the optional `STYLE_PREFIX` to steer the visual style.


In [ ]:
!pip -q install -U "diffusers>=0.30,<1" "transformers>=4.44,<5" "accelerate>=0.33,<2" safetensors huggingface_hub Pillow


## 1. Load the prompts from GitHub

Set `BRANCH` to the branch containing this notebook and `image_prompts.txt`. Public GitHub repository required for unauthenticated clone. If you have not pushed the files yet, upload `image_prompts.txt` manually to Colab and change `PROMPTS_FILE` below.


In [ ]:
import os, re, subprocess
from pathlib import Path

REPO = "https://github.com/secretrajan84-glitch/Rajan-gill.git"
BRANCH = "arena/01a0dce9-rajan-gill"
REPO_DIR = Path("/content/Rajan-gill")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)
PROMPTS_FILE = REPO_DIR / "image_prompts.txt"
text = PROMPTS_FILE.read_text(encoding="utf-8")
# Numbered prompts are separated by at least 3 blank lines; preserve the complete style suffix.
blocks = re.split(r"\n(?:[ \t]*\n){3,}", text.strip())
prompts = {}
for block in blocks:
    match = re.match(r"^(\d+)\.\s+([\s\S]+)$", block.strip())
    if match:
        number = int(match.group(1))
        if number in prompts:
            raise ValueError(f"Duplicate prompt number {number}")
        prompts[number] = match.group(2).strip()
assert len(prompts) == 182, f"Expected 182 prompts; found {len(prompts)}. Check separators in {PROMPTS_FILE}"
print(f"Loaded {len(prompts)} prompts, numbers {min(prompts)}–{max(prompts)}")


## 2. Settings and persistent output

Try `WIDTH=640, HEIGHT=360` for T4. For larger images try 768×432 only if your GPU has enough memory. Lower `STEPS` to speed up or raise it for more detail. `START`/`END` let you render a subset. The script skips existing nonempty PNGs unless `OVERWRITE=True`.


In [ ]:
from google.colab import drive
from pathlib import Path
import torch

drive.mount("/content/drive")
OUTPUT_DIR = Path("/content/drive/MyDrive/elephant_2d_images")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_ID = "Lykon/dreamshaper-8"  # public SD 1.5-based model; can be changed
START, END = 1, 182
WIDTH, HEIGHT = 640, 360  # divisible by 8; 16:9
STEPS = 25
GUIDANCE_SCALE = 7.0
SEED = 2026
OVERWRITE = False
STYLE_PREFIX = "Hand-drawn flat 2D documentary explainer cartoon, clean black outlines, simple pastel blue sky, flat muted color shapes, consistent minimal animation style. "
NEGATIVE_PROMPT = "photorealistic, photograph, 3d render, gradient, dramatic lighting, glossy, texture noise, watermark, blurry, distorted anatomy"
assert torch.cuda.is_available(), "Select Runtime > Change runtime type > T4 GPU, then rerun."
assert WIDTH % 8 == 0 and HEIGHT % 8 == 0
assert 1 <= START <= END <= 182
print("GPU:", torch.cuda.get_device_name(0), "Output:", OUTPUT_DIR)


## 3. Load model and generate

This will download model weights (several GB), then save each finished image straight to Drive. Run again after Colab disconnects to continue. Prompt `NNN` is saved as `NNN.png`.


In [ ]:
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
from PIL import Image
from tqdm.auto import tqdm
import gc

pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, safety_checker=None,
    requires_safety_checker=False,
)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to("cuda")
pipe.enable_attention_slicing()
# SD 1.5 CLIP truncates very long prompts to 77 tokens. Keep scene text first,
# and use a short style prefix; the repeated suffix may be truncated.
for number in tqdm(range(START, END + 1), desc="Images"):
    destination = OUTPUT_DIR / f"{number:03d}.png"
    if destination.exists() and destination.stat().st_size > 0 and not OVERWRITE:
        continue
    scene = prompts[number].split(' Rendered in a flat 2D vector')[0]
    prompt = scene + " " + STYLE_PREFIX
    generator = torch.Generator(device="cuda").manual_seed(SEED + number)
    try:
        image = pipe(
            prompt=prompt, negative_prompt=NEGATIVE_PROMPT,
            width=WIDTH, height=HEIGHT, num_inference_steps=STEPS,
            guidance_scale=GUIDANCE_SCALE, generator=generator,
        ).images[0]
        temporary = destination.with_suffix(".partial.png")
        image.save(temporary)
        temporary.replace(destination)  # interrupted writes won't be mistaken for complete images
        if number % 10 == 0:
            print(f"Saved through {number}: {destination}")
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        raise RuntimeError(f"GPU ran out of memory at {number}. Lower WIDTH/HEIGHT or restart runtime.")
print("Complete:", len(list(OUTPUT_DIR.glob("[0-9][0-9][0-9].png"))), "PNG files in", OUTPUT_DIR)


## 4. Preview and optional ZIP download

ZIP creation duplicates storage usage temporarily. For large sets, prefer downloading the folder from Drive instead.


In [ ]:
from IPython.display import display
from PIL import Image
files = sorted(OUTPUT_DIR.glob("[0-9][0-9][0-9].png"))
print(f"Generated: {len(files)}/182")
for f in files[:4]:
    print(f.name)
    display(Image.open(f).resize((480, 270)))


In [ ]:
# Optional: uncomment to create a ZIP in Drive (may need ~2x free space).
# import zipfile
# archive = OUTPUT_DIR.parent / "elephant_2d_images.zip"
# with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as z:
#     for f in sorted(OUTPUT_DIR.glob("[0-9][0-9][0-9].png")):
#         z.write(f, arcname=f.name)
# print(archive)
